# 마무리 SFT · 첫 gradient 진단과 결정적 학습

동일한 자체 합성48개, 원래 초안·마무리 token IDs, 시드3407·epoch순서3407/3408·Q/V LoRA·2epoch/24update를 유지합니다. **학습의 수치 실행 조건을 새로 통제하는 별도 실험**입니다. 과거493점 가중치를 그대로 복구하는 실험이 아닙니다.

먼저 실제 Qwen 첫 decoder attention의 Q/K/V와 들어오는 gradient를 고정해 기본/결정적 flag/math SDPA 역전파를 반복 비교하고, 실제 선택된 연산 이름을 기록합니다. 엄격한 설정의 첫4문항 gradient를 두 새 프로세스에서 검증한 뒤에만 전체48개 학습A/B를 진행합니다. 첫4문항 검사는 optimizer 업데이트0회입니다. 자동 SDPA 조건이 실패하면 math 조건의 첫 gradient 검사로 한 번만 대체하며, 조건 변경을 기록합니다.

학습A/B의 실제 가중치를 각각 백업한 후, 원래 첫 Accounting24개 묶음을2048·P0·V8·문항seed·MP0로 새 프로세스에서 평가합니다. 최종 가중치144개·24회gradient·첫 logits·전체 logp·24응답을 비교합니다. MMMU900은 학습에 넣지 않고 전체900 재평가도 하지 않습니다.

Colab Drive 인증/수동 업로드 없이 고정 소스·자료를 내장했습니다.5개 증분ZIP과ready marker를 사이드바에서 회수해 본인Drive에 업로드하고 실제 회수 SHA256/CRC/모든 내부 파일을 검증한 뒤 확인합니다. 마지막ZIP만으로 복구되지 않습니다. 계산최대3시간·백업확인10분·완료/실패 자동해제를 설정했습니다. 기존 탭은 닫지 않고 진행 중셀을 재실행하거나 keep-alive를 만들지 않습니다.


In [ ]:
# @title L4 · 첫 gradient 원인 진단 + 결정적 마무리SFT 두 번 + 24문항 비교
import base64,hashlib,importlib.util,json,re,threading,time,zipfile
from pathlib import Path
from IPython.display import clear_output
from google.colab import runtime
SOURCE_SHA='98f84bff5b5c8518ea1c330c919d2cbaf0ede6b883478d21f49175dcdaf718f0'
SOURCE_B64='...'
CORE_SHA='49e2126670434e0ca1960d758a2d08c8bd2664f59f57423fe55f9dcaccf5fffe'
CORE_B64='...'
BUNDLE_SHA='0bc1895db561eca1440e820bdbfb067d256ad7a847342e22c4e96f6ded56fb98'
BUNDLE_B64='...'
_source=base64.b64decode(SOURCE_B64);_core=base64.b64decode(CORE_B64);_bundle_bytes=base64.b64decode(BUNDLE_B64)
assert hashlib.sha256(_source).hexdigest()==SOURCE_SHA
assert hashlib.sha256(_core).hexdigest()==CORE_SHA
assert hashlib.sha256(_bundle_bytes).hexdigest()==BUNDLE_SHA
_boot=Path('/content/finish-sft-deterministic-bootstrap-20261118')
assert not _boot.exists(), '기존 실행 자료가 있습니다. 셀을 재실행하지 마세요.'
_boot.mkdir();_script=_boot/'deterministic.py';_script.write_bytes(_source)
(_boot/'reproduce.py').write_bytes(_core)
_bundle=_boot/'frozen_bundle.zip';_bundle.write_bytes(_bundle_bytes)
_spec=importlib.util.spec_from_file_location('deterministic_finish',_script)
_det=importlib.util.module_from_spec(_spec);_spec.loader.exec_module(_det)
_repro=_det.core
assert _det.CORE_SHA==CORE_SHA and _repro.BUNDLE_SHA==BUNDLE_SHA
_released=False
_seen={};_receipts=[]
def _release():
    global _released
    if not _released:
        _released = True
        print('REPRO_RELEASE_REQUEST', json.dumps({'unix': time.time()}), flush=True)
        runtime.unassign()
def _backup(artifacts, label):
    if label not in ('diagnosis', 'train_A', 'train_B', 'compare', 'final'):
        return
    changed = []
    updates = {}
    for f in sorted(Path(artifacts).rglob('*')):
        if not f.is_file():
            continue
        rel = f.relative_to(artifacts)
        if 'merged' in rel.parts or str(rel) == 'frozen_bundle.zip' or any((x.startswith('.') or x.endswith('.pending') for x in rel.parts)):
            continue
        assert not f.is_symlink()
        digest = _repro.sha(f)
        updates[str(rel)] = digest
        if _seen.get(str(rel)) != digest:
            changed.append((rel, f))
    if not changed:
        return
    seq = len(_receipts) + 1
    stem = f'{seq:06d}-{label}'
    manifest = {'sequence': seq, 'stage': label, 'files': {str(r): updates[str(r)] for r, _ in changed}, 'source_sha256': SOURCE_SHA, 'core_source_sha256': CORE_SHA, 'frozen_bundle_sha256': BUNDLE_SHA, 'frozen_bundle_drive_file_id': '1bZ3DIRzU2yFkdB0QZF9pSfkp3hcvKYD6', 'source_policy': 'Frozen48 objective unchanged; strict numerical controls, actual SDPA replay and first-gradient preflights; MMMU only evaluation'}
    archive = _boot / (stem + '.zip')
    with zipfile.ZipFile(archive, 'x', zipfile.ZIP_DEFLATED, compresslevel=6) as z:
        for rel, f in changed:
            z.write(f, str(rel))
        z.writestr('_checkpoint_manifest.json', json.dumps(manifest, ensure_ascii=False, indent=2) + '\n')
    with zipfile.ZipFile(archive) as z:
        assert z.testzip() is None
    data = archive.read_bytes()
    digest = hashlib.sha256(data).hexdigest()
    encoded = base64.b64encode(data).decode()
    chunks = [encoded[i:i + 65536] for i in range(0, len(encoded), 65536)]
    header = {'sequence': seq, 'stage': label, 'archive': archive.name, 'bytes': len(data), 'sha256': digest, 'chunks': len(chunks), 'payload_files': len(changed)}
    print('REPRO_CHECKPOINT_BEGIN ' + json.dumps(header, ensure_ascii=False), flush=True)
    for i, chunk in enumerate(chunks):
        print('REPRO_CHECKPOINT_DATA ' + str(seq) + ' ' + str(i) + ' ' + chunk, flush=True)
    print('REPRO_CHECKPOINT_END ' + str(seq), flush=True)
    wait_guard = threading.Timer(600, _release)
    wait_guard.daemon = True
    wait_guard.start()
    try:
        answer = input('체크섬 및 Drive 왕복 검증 완료 후 파일ID:SHA256 입력: ').strip()
        parts = answer.split(':')
        assert len(parts) == 2 and re.fullmatch('[A-Za-z0-9_-]{15,100}', parts[0]) and (parts[1] == digest)
    finally:
        wait_guard.cancel()
    receipt = header | {'drive_file_id': parts[0], 'external_verification': 'Sidebar visible output locally decoded; ZIP CRC/member SHA checked; Drive connector uploaded and fetched; actual byte SHA matched before this receipt', 'verified_unix': time.time()}
    _repro.write(Path(artifacts) / 'receipts' / f'{seq:06d}.ready.json', receipt)
    _seen.clear()
    _seen.update(updates)
    _receipts.append(receipt)
    clear_output(wait=False)
    print('REPRO_CHECKPOINT_BACKED_UP ' + json.dumps(receipt, ensure_ascii=False), flush=True)
try:
    print('DETER_SOURCE_LOCK',json.dumps({'source_sha256':SOURCE_SHA,'core_sha256':CORE_SHA,'bundle_sha256':BUNDLE_SHA}),flush=True)
    _det.launch(_bundle,'/content/finish-sft-deterministic-20261118',backup=_backup,release=_release,max_seconds=10800)
finally:
    if not _released:_release()
